# ATLAS eval on `train.csv` (README scoring)

Eval-only notebook. Same scoring as [ATLAS-Qwen2.5-7B README](../ATLAS-Qwen2.5-7B/README.md):

```python
scores = model.get_score_vectors(seq, attn, prod_temps=(1, 2, 4, 8))  # [B, 5, 5]
pred = scores[:, :, 1:].mean(-1).argmax(-1)  # argmax-prod over T
```

Loads the adapter from **zuhairsan/atlas-qwen2-5-7b** on Kaggle (not Hugging Face). The checkpoint is **5 families** (`gpt`, `claude`, `gemini`, `mistral`, `llama`). Competition `train.csv` labels are mapped to those families; `qwen` / `glm` / `kimi` rows have no family and are excluded from the accuracy denominator.

**Attach:**

1. Competition **Name That LLM** (`train.csv`)
2. Model **Qwen2.5-7B** base (`qwen-lm/qwen2.5`, variation `7b`)
3. Dataset **zuhairsan/atlas-qwen2-5-7b**

Turn **internet ON** so the install cell can `pip install bitsandbytes` (faster 4-bit on T4). If internet is off, the script falls back to fp16 + CPU offload.


## 0. Install deps


In [ ]:
# Install deps for ATLAS eval. Turn internet ON for this notebook (eval only, not a competition submit).
import subprocess, sys

pkgs = ["peft", "accelerate", "safetensors", "bitsandbytes"]
print("installing", pkgs, flush=True)
r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", *pkgs],
    capture_output=True, text=True,
)
print((r.stdout or "")[-1500:], flush=True)
if r.returncode != 0:
    print((r.stderr or "")[-2000:], flush=True)
    # Offline fallback: local manylinux wheel if attached
    import glob
    wheels = sorted(glob.glob("/kaggle/input/**/bitsandbytes*manylinux*.whl", recursive=True))
    if wheels:
        print("offline wheel", wheels[0], flush=True)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-deps", "--force-reinstall", wheels[0]])
    else:
        print("bitsandbytes pip failed and no wheel found; eval will use fp16+CPU offload", flush=True)
else:
    import bitsandbytes as bnb
    print("bitsandbytes", getattr(bnb, "__version__", "?"), flush=True)


## 1. Write eval script (README-style ATLAS scoring)


In [ ]:
%%writefile eval_atlas_train.py
"""Eval ATLAS (5-family) on competition train.csv. README-style scoring."""
import importlib.util as _iu
import glob
import json
import os
import sys
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

_fs = _iu.find_spec
_iu.find_spec = lambda name, *a, **k: None if name.split(".")[0] == "torchao" else _fs(name, *a, **k)

from peft import LoraConfig, get_peft_model
import peft.import_utils as _pi

_pi.is_torchao_available = lambda *a, **k: False

try:
    from peft.tuners.lora.layer import Linear as LoraLinear
except ImportError:
    from peft.tuners.lora import Linear as LoraLinear

try:
    from peft import set_peft_model_state_dict
except ImportError:
    from peft.utils.save_and_load import set_peft_model_state_dict

A2_B_INIT_STD = 1e-4
PROD_TEMPS = (1.0, 2.0, 4.0, 8.0)
SOURCES = ["gpt", "claude", "gemini", "mistral", "llama"]

# Competition labels -> ATLAS family (None = no matching family in the checkpoint)
MODEL_TO_FAMILY = {
    "gpt-5.5": "gpt",
    "gpt-5.6-luna": "gpt",
    "gpt-5.6-sol": "gpt",
    "gpt-5.6-terra": "gpt",
    "gpt-6-astra": "gpt",
    "claude-fable-5": "claude",
    "claude-opus-4-6": "claude",
    "claude-opus-4-8": "claude",
    "claude-opus-5": "claude",
    "claude-sonnet-5": "claude",
    "gemini-3.1-pro-preview": "gemini",
    "gemini-3.6-flash": "gemini",
    "gemini-3.7-flash": "gemini",
    "gemini-3.8-flash": "gemini",
    "glm-5.3": None,
    "glm-5.3-flash": None,
    "kimi-k3": None,
    "qwen3.8-max": None,
}


class _LoraRoute:
    __slots__ = ("index", "uniform_k")

    def __init__(self):
        self.index = None
        self.uniform_k = None

    def set(self, index, uniform_k):
        self.index, self.uniform_k = index, uniform_k


class PerSourceLoraLinear(LoraLinear):
    @classmethod
    def install(cls, layer, num_sources, route, init_std):
        adapters = list(layer.lora_A.keys())
        if len(adapters) != 1:
            raise RuntimeError(f"per-source B expects one adapter, got {adapters}")
        name = adapters[0]
        out_f, r = layer.lora_B[name].weight.shape
        layer.__class__ = cls
        layer._route = route
        layer._adapter_name = name
        w = torch.randn(
            num_sources, out_f, r,
            dtype=layer.lora_B[name].weight.dtype,
            device=layer.lora_B[name].weight.device,
        ) * init_std
        layer.lora_B_k = nn.Parameter(w)
        layer._freeze_shadow_b()
        return layer

    def _freeze_shadow_b(self):
        with torch.no_grad():
            self.lora_B[self._adapter_name].weight.zero_()
        self.lora_B[self._adapter_name].weight.requires_grad_(False)

    def set_adapter(self, adapter_names, inference_mode=False):
        super().set_adapter(adapter_names, inference_mode=inference_mode)
        self._freeze_shadow_b()

    def forward(self, x, *args, **kwargs):
        if hasattr(self, "_check_forward_args"):
            self._check_forward_args(x, *args, **kwargs)
        kwargs.pop("adapter_names", None)
        if getattr(self, "disable_adapters", False):
            if getattr(self, "merged", False):
                self.unmerge()
            return self.base_layer(x, *args, **kwargs)
        result = self.base_layer(x, *args, **kwargs)
        out_dtype = result.dtype
        name = self._adapter_name
        lora_A = self.lora_A[name]
        dropout = self.lora_dropout[name]
        if hasattr(self, "_cast_input_dtype"):
            x = self._cast_input_dtype(x, lora_A.weight.dtype)
        else:
            x = x.to(lora_A.weight.dtype)
        h = lora_A(dropout(x))
        B = self.lora_B_k
        k = self._route.uniform_k
        if k is not None:
            delta = F.linear(h, B[k])
        else:
            idx = self._route.index
            if idx is None or idx.numel() != h.shape[0]:
                raise RuntimeError("per-source LoRA B: row->source route is missing")
            Bn = B.index_select(0, idx)
            delta = torch.bmm(h, Bn.transpose(1, 2))
        return (result + delta.to(out_dtype) * self.scaling[name]).to(out_dtype)


def masked_mean(x, mask):
    return (x * mask).sum(dim=-1) / mask.sum(dim=-1).clamp(min=1.0)


def doc_score(theta_lp, mask):
    return masked_mean(theta_lp, mask.unsqueeze(1))


def prod_readout(theta_lp, mask, temperature):
    m = mask > 0.5
    lp = theta_lp.float()
    gap = lp.unsqueeze(2) - lp.unsqueeze(1)
    v = torch.exp(-(torch.relu(-gap) / float(temperature)).sum(dim=2))
    v = v * m.unsqueeze(1)
    return v.sum(dim=2) / m.sum(dim=1, keepdim=True).clamp(min=1)


def try_bitsandbytes():
    try:
        import bitsandbytes as bnb  # noqa: F401
        print(f"bitsandbytes: {getattr(bnb, '__version__', '?')}", flush=True)
        return True
    except Exception as e:
        print(f"bitsandbytes missing ({type(e).__name__}); fp16 + CPU offload", flush=True)
        return False


class ATLASModel(nn.Module):
    """Same recipe as ATLAS README: LoRA + per-source B_k + control embeddings."""

    def __init__(self, base_path, source_names, lora_r=16, lora_alpha=32, lora_dropout=0.05, prefix_len=1, device="cuda"):
        super().__init__()
        self.source_names = list(source_names)
        self.num_sources = len(self.source_names)
        self.prefix_len = int(prefix_len)
        self.tail_chunks = 4
        self.device_ = torch.device(device)

        self.tokenizer = AutoTokenizer.from_pretrained(base_path, local_files_only=True)
        self.tokenizer.padding_side = "right"
        self.tokenizer.model_max_length = int(1e9)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        use_4bit = try_bitsandbytes()
        load_kw = dict(local_files_only=True, low_cpu_mem_usage=True, attn_implementation="sdpa")
        if use_4bit:
            from transformers import BitsAndBytesConfig
            from peft import prepare_model_for_kbit_training

            load_kw["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
                bnb_4bit_compute_dtype=torch.float16,
            )
            load_kw["device_map"] = {"": self.device_}
        else:
            load_kw["device_map"] = "auto"
            load_kw["max_memory"] = {0: "11GiB", "cpu": "48GiB"}

        try:
            base = AutoModelForCausalLM.from_pretrained(base_path, dtype=torch.float16, **load_kw)
        except TypeError:
            base = AutoModelForCausalLM.from_pretrained(base_path, torch_dtype=torch.float16, **load_kw)
        base.config.use_cache = False
        if use_4bit:
            base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=False)

        self.model = get_peft_model(
            base,
            LoraConfig(
                r=lora_r,
                lora_alpha=lora_alpha,
                lora_dropout=lora_dropout,
                target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
                bias="none",
                task_type="CAUSAL_LM",
            ),
        )
        self._route = _LoraRoute()
        self._per_source_layers = []
        for name, mod in self.model.named_modules():
            if isinstance(mod, LoraLinear) and not isinstance(mod, PerSourceLoraLinear):
                PerSourceLoraLinear.install(mod, self.num_sources, self._route, A2_B_INIT_STD)
                self._per_source_layers.append((name, mod))
        if not self._per_source_layers:
            raise RuntimeError("no LoRA layers found to convert to per-source B")

        emb = self.model.get_input_embeddings().weight.detach().float()
        mean = emb.mean(dim=0)
        init = mean.view(1, 1, -1).repeat(self.num_sources, self.prefix_len, 1)
        self.control_embeddings = nn.Parameter(init.to(device=self.device_, dtype=torch.float32))
        free, total = torch.cuda.mem_get_info(self.device_)
        print(
            f"base ready ({'4-bit' if use_4bit else 'fp16+offload'}), "
            f"free {free / 1024 ** 3:.2f}/{total / 1024 ** 3:.2f} GiB",
            flush=True,
        )

    def _decoder(self):
        base = self.model.get_base_model()
        return base.model if hasattr(base, "model") else base

    def _lm_head(self, hidden):
        return self.model.get_output_embeddings()(hidden)

    def _theta_pass_single(self, input_ids, attention_mask, labels):
        bsz = input_ids.shape[0]
        pfx = self.prefix_len
        tok_emb = self.model.get_input_embeddings()(input_ids)
        ctrl = self.control_embeddings.index_select(0, labels).to(dtype=tok_emb.dtype)
        inputs_emb = torch.cat([ctrl, tok_emb], dim=1)
        full_attn = torch.cat([attention_mask.new_ones(bsz, pfx), attention_mask], dim=1)
        self._route.set(labels, None)
        hidden = self._decoder()(inputs_embeds=inputs_emb, attention_mask=full_attn).last_hidden_state
        hid = hidden[:, pfx:-1, :]
        targets = input_ids[:, 1:].unsqueeze(-1)
        del hidden

        def tail(h_c, t_c):
            log_s = F.log_softmax(self._lm_head(h_c).float(), dim=-1)
            return log_s.gather(-1, t_c).squeeze(-1)

        n = max(1, min(self.tail_chunks, hid.size(1)))
        parts = [tail(h_c, t_c) for h_c, t_c in zip(hid.chunk(n, dim=1), targets.chunk(n, dim=1))]
        return torch.cat(parts, dim=1)

    @torch.no_grad()
    def get_score_vectors(self, input_ids, attention_mask, prod_temps=()):
        """README readout. Returns [B, K, 1 + len(prod_temps)]."""
        chunks = []
        for k in range(self.num_sources):
            lab = torch.full((input_ids.size(0),), k, device=input_ids.device, dtype=torch.long)
            chunks.append(self._theta_pass_single(input_ids, attention_mask, lab).float())
        theta_lp = torch.stack(chunks, dim=1)
        loss_mask = attention_mask[:, 1:].float()
        cols = [doc_score(theta_lp, loss_mask)]
        cols += [prod_readout(theta_lp, loss_mask, t) for t in prod_temps]
        return torch.stack(cols, dim=-1)

    def load_weights(self, ckpt_dir):
        extra = torch.load(os.path.join(ckpt_dir, "atlas_extra.pt"), map_location="cpu", weights_only=False)
        saved_names = list(extra["source_names"])
        if saved_names != self.source_names:
            raise ValueError(f"checkpoint sources {saved_names} != {self.source_names}")
        st_path = os.path.join(ckpt_dir, "adapter_model.safetensors")
        if os.path.exists(st_path):
            from safetensors.torch import load_file
            state = load_file(st_path)
        else:
            state = torch.load(os.path.join(ckpt_dir, "adapter_model.bin"), map_location="cpu", weights_only=False)
        state = {k: v for k, v in state.items() if "lora_B_k" not in k}
        set_peft_model_state_dict(self.model, state)
        saved = extra["control_embeddings"]
        if saved.dim() == 2:
            saved = saved.unsqueeze(1)
        if saved.shape != self.control_embeddings.shape:
            self.control_embeddings = nn.Parameter(saved.to(device=self.device_, dtype=torch.float32))
            self.prefix_len = int(saved.shape[1])
        else:
            with torch.no_grad():
                self.control_embeddings.copy_(saved.to(self.control_embeddings.dtype))
        saved_b = extra["lora_B_k"]
        with torch.no_grad():
            for n, m in self._per_source_layers:
                m.lora_B_k.copy_(saved_b[n].to(device=m.lora_B_k.device, dtype=m.lora_B_k.dtype))
                m._freeze_shadow_b()
        print(f"loaded weights from {ckpt_dir}", flush=True)
        return self


def find_adapter(hint=""):
    roots = [hint] if hint else []
    roots += [
        "/kaggle/input/datasets/zuhairsan/atlas-qwen2-5-7b",
        "/kaggle/input/atlas-qwen2-5-7b",
        "/kaggle/input",
        "ATLAS-Qwen2.5-7B",
        "../ATLAS-Qwen2.5-7B",
    ]
    hits = []
    for root in roots:
        if not root or not os.path.exists(root):
            continue
        for dirpath, _, files in os.walk(root):
            if "atlas_extra.pt" in files and (
                "adapter_model.safetensors" in files or "adapter_model.bin" in files
            ):
                hits.append(dirpath)
        if hits and root != "/kaggle/input":
            break
    if not hits:
        raise FileNotFoundError("ATLAS adapter not found (need atlas_extra.pt + adapter_model.*)")
    hits.sort(key=lambda p: (0 if "atlas" in p.lower() else 1, len(p)))
    return hits[0]


def find_qwen25():
    hits = []
    for cfg_path in sorted(glob.glob("/kaggle/input/**/config.json", recursive=True)):
        try:
            j = json.load(open(cfg_path))
        except Exception:
            continue
        if j.get("model_type") != "qwen2":
            continue
        if j.get("hidden_size") != 3584 or j.get("num_hidden_layers") != 28:
            continue
        folder = os.path.dirname(cfg_path)
        low = folder.lower()
        if "qwen2.5" not in low and "qwen2-5" not in low and "qwen2_5" not in low:
            continue
        if any(bad in low for bad in ("awq", "gptq", "int4", "int8")):
            continue
        hits.append(folder)
    if not hits:
        raise FileNotFoundError("Qwen2.5-7B not found under /kaggle/input")
    hits.sort(key=lambda p: (1 if "instruct" in p.lower() else 0, len(p)))
    return hits[0]


def find_csv(name):
    for p in [
        f"/kaggle/input/competitions/name-that-llm/{name}",
        f"/kaggle/input/name-that-llm/{name}",
        f"data/{name}",
        f"../data/{name}",
    ]:
        if os.path.exists(p):
            return p
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    if hits:
        return hits[0]
    raise FileNotFoundError(name)


def bal_acc(y_true, y_pred, classes):
    scores = []
    for c in classes:
        m = y_true == c
        if m.any():
            scores.append(float((y_pred[m] == c).mean()))
    return float(np.mean(scores)) if scores else float("nan")


def main():
    cfg = json.loads(sys.argv[1]) if len(sys.argv) > 1 else {}
    t0 = time.time()
    max_len = int(cfg.get("max_len", 512))
    batch_size = int(cfg.get("batch_size", 1))
    limit = int(cfg.get("limit", 0))  # 0 = all rows

    adapter_dir = cfg.get("adapter_dir") or find_adapter(cfg.get("adapter_hint", ""))
    base_dir = cfg.get("model_dir") or find_qwen25()
    train_csv = cfg.get("train_csv") or find_csv("train.csv")
    print("adapter", adapter_dir, flush=True)
    print("base", base_dir, flush=True)
    print("train", train_csv, flush=True)

    extra = torch.load(os.path.join(adapter_dir, "atlas_extra.pt"), map_location="cpu", weights_only=False)
    sources = list(extra.get("source_names", SOURCES))
    print(f"checkpoint sources: {sources}", flush=True)

    lora_r, lora_alpha, lora_dropout, prefix_len = 16, 32, 0.05, 1
    acfg = os.path.join(adapter_dir, "adapter_config.json")
    if os.path.exists(acfg):
        ac = json.load(open(acfg))
        lora_r = int(ac.get("r", lora_r))
        lora_alpha = int(ac.get("lora_alpha", lora_alpha))
        lora_dropout = float(ac.get("lora_dropout", lora_dropout))
    if extra["control_embeddings"].dim() == 3:
        prefix_len = int(extra["control_embeddings"].shape[1])

    model = ATLASModel(
        base_dir, sources,
        lora_r=lora_r, lora_alpha=lora_alpha, lora_dropout=lora_dropout,
        prefix_len=prefix_len, device="cuda:0",
    )
    model.load_weights(adapter_dir)
    model.eval()

    train = pd.read_csv(train_csv)
    train["text"] = train["text"].astype(str)
    train["model"] = train["model"].astype(str)
    train["family"] = train["model"].map(MODEL_TO_FAMILY)
    if limit > 0:
        train = train.iloc[:limit].copy()
        print(f"limit={limit}", flush=True)
    print(f"rows {len(train)}  with family {train['family'].notna().sum()}  no-family {train['family'].isna().sum()}", flush=True)

    tok = model.tokenizer
    tokens = tok(
        train["text"].tolist(),
        add_special_tokens=True,
        truncation=True,
        max_length=max_len,
    )["input_ids"]
    tokens = [t if len(t) >= 2 else (t + [tok.eos_token_id] * (2 - len(t))) for t in tokens]

    K = len(sources)
    prod = np.zeros((len(train), K), np.float32)
    logprob = np.zeros((len(train), K), np.float32)
    order = np.argsort([len(t) for t in tokens], kind="stable")
    device = model.device_

    with torch.inference_mode():
        for start in range(0, len(order), batch_size):
            pos = order[start : start + batch_size]
            batch = tok.pad({"input_ids": [tokens[i] for i in pos]}, return_tensors="pt")
            ids = batch["input_ids"].to(device)
            attn = batch["attention_mask"].to(device)
            with torch.autocast("cuda", dtype=torch.float16):
                scores = model.get_score_vectors(ids, attn, prod_temps=PROD_TEMPS)  # [B,K,5]
            # col 0 = mean logprob; cols 1.. = prod temps
            logprob[pos] = scores[:, :, 0].float().cpu().numpy()
            prod[pos] = scores[:, :, 1:].mean(-1).float().cpu().numpy()
            if (start // batch_size) % 20 == 0:
                print(
                    f"scored {min(start + batch_size, len(order))}/{len(order)} ({time.time() - t0:.0f}s)",
                    flush=True,
                )
            del scores, ids, attn

    pred_idx = prod.argmax(1)
    pred_fam = np.array(sources)[pred_idx]
    true_fam = train["family"].to_numpy()

    # Fair eval: only rows whose true label maps to an ATLAS family
    has_fam = train["family"].notna().to_numpy()
    y_true = true_fam[has_fam]
    y_pred = pred_fam[has_fam]
    n_ok = int((y_true == y_pred).sum())
    n = int(has_fam.sum())
    acc = n_ok / n if n else float("nan")
    bacc = bal_acc(y_true, y_pred, SOURCES)

    print("\n=== ATLAS family eval on train.csv (README argmax-prod) ===", flush=True)
    print(f"rows with a family label: {n} / {len(train)}", flush=True)
    print(f"correct: {n_ok} / {n}  accuracy={acc:.4f}  balanced_acc={bacc:.4f}", flush=True)
    print("\nper-family recall:", flush=True)
    for fam in SOURCES:
        m = y_true == fam
        if m.any():
            print(f"  {fam:8s}  {int((y_pred[m] == fam).sum()):4d}/{int(m.sum()):4d}  {(y_pred[m] == fam).mean():.3f}", flush=True)
    print("\nconfusion (true \\ pred):", flush=True)
    cm = pd.crosstab(pd.Series(y_true, name="true"), pd.Series(y_pred, name="pred"), dropna=False)
    print(cm.reindex(index=SOURCES, columns=SOURCES, fill_value=0).to_string(), flush=True)

    # Rows with no family (qwen/glm/kimi): what did ATLAS guess?
    no_fam = ~has_fam
    if no_fam.any():
        print("\nno-family rows (qwen/glm/kimi) — ATLAS still emits a family guess:", flush=True)
        print(pd.Series(pred_fam[no_fam]).value_counts().to_string(), flush=True)
        print(
            "These cannot be 'correct' under the 5-family checkpoint; "
            f"{int(no_fam.sum())} rows excluded from accuracy.",
            flush=True,
        )

    out = train[["id", "model"]].copy()
    out["true_family"] = true_fam
    out["pred_family"] = pred_fam
    out["correct"] = (out["true_family"] == out["pred_family"]) & out["true_family"].notna()
    out.to_csv("atlas_train_eval.csv", index=False)
    np.savez_compressed(
        "atlas_train_scores.npz",
        ids=train["id"].to_numpy(),
        model=train["model"].to_numpy(),
        true_family=true_fam.astype(object),
        pred_family=pred_fam,
        prod=prod,
        logprob=logprob,
        sources=np.array(sources),
    )
    with open("atlas_train_eval.json", "w") as f:
        json.dump(
            {
                "n_total": int(len(train)),
                "n_with_family": n,
                "n_correct": n_ok,
                "accuracy": acc,
                "balanced_accuracy": bacc,
                "sources": sources,
                "secs": time.time() - t0,
            },
            f,
            indent=2,
        )
    print(f"\nwrote atlas_train_eval.csv / .json / atlas_train_scores.npz  ({time.time() - t0:.0f}s)", flush=True)


if __name__ == "__main__":
    main()


## 2. Run eval on train.csv

Reports how many family predictions match. Writes `atlas_train_eval.csv` / `.json` and score matrices.


In [ ]:
import json
import pathlib
import subprocess
import sys
import time

import pandas as pd

assert pathlib.Path("eval_atlas_train.py").exists(), "Run the writefile cell first"

cfg = dict(
    adapter_hint="/kaggle/input/datasets/zuhairsan/atlas-qwen2-5-7b",
    max_len=512,
    batch_size=1,
    limit=0,   # set e.g. 200 for a quick smoke; 0 = all 2699 rows
)

t0 = time.time()
log_path = "eval_atlas.log"
proc = subprocess.Popen(
    [sys.executable, "eval_atlas_train.py", json.dumps(cfg)],
    stdout=open(log_path, "w"),
    stderr=subprocess.STDOUT,
    env=dict(**{**__import__("os").environ, "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}),
)
proc.wait()
print(open(log_path, encoding="utf-8", errors="replace").read()[-12000:])
if proc.returncode != 0:
    raise SystemExit(f"eval failed rc={proc.returncode}")

summary = json.load(open("atlas_train_eval.json"))
print("summary:", json.dumps(summary, indent=2))
print(f"wall {time.time() - t0:.0f}s")
print(f"correct {summary['n_correct']} / {summary['n_with_family']}  "
      f"acc={summary['accuracy']:.4f}  bal_acc={summary['balanced_accuracy']:.4f}")
